# Práctica de PySpark — Window, Arrays, JSON y SQL

Este notebook contiene **10 ejercicios ** para repasar:

- **11. Window Functions**
- **12. Array Functions**
- **13. JSON Functions**
- **14. Running SQL Queries**

Incluye:
- enunciado,
- solución en PySpark,
- y una **salida esperada** orientativa en cada ejercicio.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("practica-window-array-json-sql").getOrCreate()

ventas_data = [
    (1, "Norte", "Ana", "2024-01-03", 120.0),
    (2, "Norte", "Ana", "2024-01-10", 90.0),
    (3, "Norte", "Luis", "2024-01-05", 150.0),
    (4, "Sur", "Marta", "2024-01-02", 200.0),
    (5, "Sur", "Marta", "2024-01-08", 50.0),
    (6, "Sur", "Juan", "2024-01-06", 80.0)
]

ventas_schema = StructType([
    StructField("venta_id", IntegerType(), False),
    StructField("zona", StringType(), True),
    StructField("vendedor", StringType(), True),
    StructField("fecha_str", StringType(), True),
    StructField("importe", DoubleType(), True)
])

ventas_df = spark.createDataFrame(ventas_data, ventas_schema).withColumn(
    "fecha", F.to_date("fecha_str", "yyyy-MM-dd")
)

productos_data = [
    (1, "Portátil", ["electronica", "oficina", "premium"], [999.0, 949.0, 899.0]),
    (2, "Ratón", ["electronica", "accesorios"], [25.0, 22.5]),
    (3, "Mesa", ["hogar", "oficina"], [199.0, 179.0, 169.0])
]

productos_schema = StructType([
    StructField("producto_id", IntegerType(), False),
    StructField("producto", StringType(), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("precios_historicos", ArrayType(DoubleType()), True)
])

productos_df = spark.createDataFrame(productos_data, productos_schema)

eventos_data = [
    (1, '{"usuario":"ana","ciudad":"Madrid","edad":28,"compra":{"producto":"teclado","importe":45.5}}'),
    (2, '{"usuario":"luis","ciudad":"Valencia","edad":35,"compra":{"producto":"monitor","importe":199.99}}'),
    (3, '{"usuario":"marta","ciudad":"Sevilla","edad":31,"compra":{"producto":"raton","importe":22.0}}')
]

eventos_schema = StructType([
    StructField("evento_id", IntegerType(), False),
    StructField("json_str", StringType(), True)
])

eventos_df = spark.createDataFrame(eventos_data, eventos_schema)

ventas_df.show()
productos_df.show(truncate=False)
eventos_df.show(truncate=False)


+--------+-----+--------+----------+-------+----------+
|venta_id| zona|vendedor| fecha_str|importe|     fecha|
+--------+-----+--------+----------+-------+----------+
|       1|Norte|     Ana|2024-01-03|  120.0|2024-01-03|
|       2|Norte|     Ana|2024-01-10|   90.0|2024-01-10|
|       3|Norte|    Luis|2024-01-05|  150.0|2024-01-05|
|       4|  Sur|   Marta|2024-01-02|  200.0|2024-01-02|
|       5|  Sur|   Marta|2024-01-08|   50.0|2024-01-08|
|       6|  Sur|    Juan|2024-01-06|   80.0|2024-01-06|
+--------+-----+--------+----------+-------+----------+

+-----------+--------+-------------------------------+---------------------+
|producto_id|producto|tags                           |precios_historicos   |
+-----------+--------+-------------------------------+---------------------+
|1          |Portátil|[electronica, oficina, premium]|[999.0, 949.0, 899.0]|
|2          |Ratón   |[electronica, accesorios]      |[25.0, 22.5]         |
|3          |Mesa    |[hogar, oficina]               |

## Ejercicio 1. Window básica: `row_number`

Asigna un número de fila por `zona`, ordenando por `importe` de mayor a menor.
Muestra:
- `zona`
- `vendedor`
- `importe`
- `fila_zona`


In [ ]:
# Importamos Window si no se ha importado antes
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Definimos la ventana: particionada por zona y ordenada por importe descendente
window_z_imp = Window.partitionBy("zona").orderBy(F.desc("importe"))

# Aplicamos row_number
df_ej1 = ventas_df.withColumn("fila_zona", F.row_number().over(window_z_imp)) \
                  .select("zona", "vendedor", "importe", "fila_zona")

df_ej1.show()


+-----+--------+-------+---------+
| zona|vendedor|importe|fila_zona|
+-----+--------+-------+---------+
|Norte|    Luis|  150.0|        1|
|Norte|     Ana|  120.0|        2|
|Norte|     Ana|   90.0|        3|
|  Sur|   Marta|  200.0|        1|
|  Sur|    Juan|   80.0|        2|
|  Sur|   Marta|   50.0|        3|
+-----+--------+-------+---------+



## Ejercicio 2. Window básica: `rank` y `dense_rank`

Calcula `rank` y `dense_rank` del importe dentro de cada zona.


In [ ]:
# Reutilizamos la ventana del ejercicio anterior
window_z_imp = Window.partitionBy("zona").orderBy(F.desc("importe"))

df_ej2 = ventas_df.withColumn("rank_importe", F.rank().over(window_z_imp)) \
                  .withColumn("dense_rank_importe", F.dense_rank().over(window_z_imp)) \
                  .select("zona", "vendedor", "importe", "rank_importe", "dense_rank_importe")

df_ej2.show()


+-----+--------+-------+------------+------------------+
| zona|vendedor|importe|rank_importe|dense_rank_importe|
+-----+--------+-------+------------+------------------+
|Norte|    Luis|  150.0|           1|                 1|
|Norte|     Ana|  120.0|           2|                 2|
|Norte|     Ana|   90.0|           3|                 3|
|  Sur|   Marta|  200.0|           1|                 1|
|  Sur|    Juan|   80.0|           2|                 2|
|  Sur|   Marta|   50.0|           3|                 3|
+-----+--------+-------+------------+------------------+



## Ejercicio 3. Window con acumulado

Calcula un acumulado de `importe` por `vendedor`, ordenado por fecha.


In [ ]:
# Definimos la ventana: particionada por vendedor y ordenada por fecha
window_vend_fecha = Window.partitionBy("vendedor").orderBy("fecha")

# Suma acumulada utilizando over()
df_ej3 = ventas_df.withColumn("acumulado_vendedor", F.sum("importe").over(window_vend_fecha)) \
                  .select("vendedor", "fecha_str", "importe", "acumulado_vendedor")

df_ej3.show()


+--------+----------+-------+------------------+
|vendedor| fecha_str|importe|acumulado_vendedor|
+--------+----------+-------+------------------+
|     Ana|2024-01-03|  120.0|             120.0|
|     Ana|2024-01-10|   90.0|             210.0|
|    Juan|2024-01-06|   80.0|              80.0|
|    Luis|2024-01-05|  150.0|             150.0|
|   Marta|2024-01-02|  200.0|             200.0|
|   Marta|2024-01-08|   50.0|             250.0|
+--------+----------+-------+------------------+



## Ejercicio 4. Window con `rowsBetween`

Calcula la media móvil de 2 filas:
- fila actual
- fila anterior

Por `vendedor` y ordenado por fecha.


In [ ]:
# Definimos la ventana con el rango de filas: la anterior y la actual
window_movil = Window.partitionBy("vendedor") \
                     .orderBy("fecha") \
                     .rowsBetween(Window.currentRow - 1, Window.currentRow)

df_ej4 = ventas_df.withColumn("media_movil_2", F.mean("importe").over(window_movil)) \
                  .select("vendedor", "fecha_str", "importe", "media_movil_2")

df_ej4.show()   


+--------+----------+-------+-------------+
|vendedor| fecha_str|importe|media_movil_2|
+--------+----------+-------+-------------+
|     Ana|2024-01-03|  120.0|        120.0|
|     Ana|2024-01-10|   90.0|        105.0|
|    Juan|2024-01-06|   80.0|         80.0|
|    Luis|2024-01-05|  150.0|        150.0|
|   Marta|2024-01-02|  200.0|        200.0|
|   Marta|2024-01-08|   50.0|        125.0|
+--------+----------+-------+-------------+



## Ejercicio 5. Arrays: crear y manipular arrays

En `ventas_df`, crea una columna array llamada `resumen_array` con:
- `zona`
- `vendedor`
- `importe` convertido a string

Después calcula también el tamaño del array.


In [ ]:
df_ej5 = ventas_df.withColumn("resumen_array", F.array(F.col("zona"), F.col("vendedor"), F.col("importe").cast("string"))) \
                  .withColumn("tam_array", F.size("resumen_array")) \
                  .select("venta_id", "resumen_array", "tam_array")

df_ej5.show(truncate=False)

+--------+--------------------+---------+
|venta_id|resumen_array       |tam_array|
+--------+--------------------+---------+
|1       |[Norte, Ana, 120.0] |3        |
|2       |[Norte, Ana, 90.0]  |3        |
|3       |[Norte, Luis, 150.0]|3        |
|4       |[Sur, Marta, 200.0] |3        |
|5       |[Sur, Marta, 50.0]  |3        |
|6       |[Sur, Juan, 80.0]   |3        |
+--------+--------------------+---------+



## Ejercicio 6. Arrays: elementos y modificación

En `productos_df`:
1. Extrae el primer tag y el último precio histórico.
2. Añade un nuevo tag llamado `nuevo`.


In [ ]:
df_ej6 = productos_df.withColumn("primer_tag", F.col("tags").getItem(0)) \
                     .withColumn("ultimo_precio", F.element_at(F.col("precios_historicos"), -1)) \
                     .withColumn("tags_actualizados", F.array_union(F.col("tags"), F.array(F.lit("nuevo")))) \
                     .select("producto", "primer_tag", "ultimo_precio", "tags_actualizados")

df_ej6.show(truncate=False)


+--------+-----------+-------------+--------------------------------------+
|producto|primer_tag |ultimo_precio|tags_actualizados                     |
+--------+-----------+-------------+--------------------------------------+
|Portátil|electronica|899.0        |[electronica, oficina, premium, nuevo]|
|Ratón   |electronica|22.5         |[electronica, accesorios, nuevo]      |
|Mesa    |hogar      |169.0        |[hogar, oficina, nuevo]               |
+--------+-----------+-------------+--------------------------------------+



## Ejercicio 7. Arrays to Rows y Rows to Array

1. Explota `tags` para tener una fila por tag.
2. Luego agrupa otra vez por `producto` y reconstruye un array de tags únicos.


In [ ]:
# 1. Explota `tags` para tener una fila por tag
df_ej7_exploded = productos_df.select("producto", F.explode("tags").alias("tag"))
df_ej7_exploded.show()

# 2. Agrupa por producto y reconstruye un array de tags únicos
df_ej7_grouped = df_ej7_exploded.groupBy("producto") \
                                .agg(F.collect_set("tag").alias("tags_reconstruidos"))
df_ej7_grouped.show(truncate=False)


+--------+-----------+
|producto|        tag|
+--------+-----------+
|Portátil|electronica|
|Portátil|    oficina|
|Portátil|    premium|
|   Ratón|electronica|
|   Ratón| accesorios|
|    Mesa|      hogar|
|    Mesa|    oficina|
+--------+-----------+

+--------+-------------------------------+
|producto|tags_reconstruidos             |
+--------+-------------------------------+
|Mesa    |[hogar, oficina]               |
|Portátil|[premium, oficina, electronica]|
|Ratón   |[accesorios, electronica]      |
+--------+-------------------------------+



## Ejercicio 8. JSON: extracción simple y múltiple

En `eventos_df`:
1. Extrae `usuario` y `ciudad` con `get_json_object`.
2. Extrae varios campos a la vez con `json_tuple`.


In [ ]:
# 1. Extraer usuario y ciudad con get_json_object
df_ej8_1 = eventos_df.withColumn("usuario", F.get_json_object("json_str", "$.usuario")) \
                     .withColumn("ciudad", F.get_json_object("json_str", "$.ciudad")) \
                     .select("evento_id", "json_str", "usuario", "ciudad")
df_ej8_1.show(truncate=False)

# 2. Extraer varios campos a la vez con json_tuple
df_ej8_2 = eventos_df.select("evento_id", F.json_tuple("json_str", "usuario", "ciudad", "edad"))
df_ej8_2.show(truncate=False)


+---------+-------------------------------------------------------------------------------------------------+-------+--------+
|evento_id|json_str                                                                                         |usuario|ciudad  |
+---------+-------------------------------------------------------------------------------------------------+-------+--------+
|1        |{"usuario":"ana","ciudad":"Madrid","edad":28,"compra":{"producto":"teclado","importe":45.5}}     |ana    |Madrid  |
|2        |{"usuario":"luis","ciudad":"Valencia","edad":35,"compra":{"producto":"monitor","importe":199.99}}|luis   |Valencia|
|3        |{"usuario":"marta","ciudad":"Sevilla","edad":31,"compra":{"producto":"raton","importe":22.0}}    |marta  |Sevilla |
+---------+-------------------------------------------------------------------------------------------------+-------+--------+

+---------+-----+--------+---+
|evento_id|c0   |c1      |c2 |
+---------+-----+--------+---+
|1        |ana  |

## Ejercicio 9. JSON: `from_json`, inferencia, flattening y `to_json`

1. Convierte el JSON a struct con `from_json`.
2. Aplana los campos anidados de `compra`.
3. Vuelve a convertir la struct `compra` a JSON string.


In [ ]:
# Inferimos el esquema leyendo los strings como RDD (también puedes definir un StructType manual)
schema_json = spark.read.json(eventos_df.rdd.map(lambda r: r.json_str)).schema

# 1. Convertir JSON a struct
# 2. Aplanar los campos
# 3. Convertir a string JSON la columna compra
df_ej9 = eventos_df.withColumn("parsed_json", F.from_json(F.col("json_str"), schema_json)) \
                   .select(
                       "evento_id",
                       F.col("parsed_json.usuario").alias("usuario"),
                       F.col("parsed_json.compra.producto").alias("producto"),
                       F.col("parsed_json.compra.importe").alias("importe"),
                       F.to_json(F.col("parsed_json.compra")).alias("compra_json")
                   )

df_ej9.show(truncate=False)
print(schema_json.simpleString())



+---------+-------+--------+-------+---------------------------------------+
|evento_id|usuario|producto|importe|compra_json                            |
+---------+-------+--------+-------+---------------------------------------+
|1        |ana    |teclado |45.5   |{"producto":"teclado","importe":45.5}  |
|2        |luis   |monitor |199.99 |{"producto":"monitor","importe":199.99}|
|3        |marta  |raton   |22.0   |{"producto":"raton","importe":22.0}    |
+---------+-------+--------+-------+---------------------------------------+

struct<ciudad:string,compra:struct<importe:double,producto:string>,edad:bigint,usuario:string>


## Ejercicio 10. SQL con y sin temp view

### Parte A
Crea una **temp view** de `ventas_df` y consulta:
- total de importe por zona.

### Parte B
Haz una consulta SQL **sin temp view** usando una subconsulta creada con `spark.sql` a partir de una vista temporal intermedia.


In [ ]:
# Parte A: Crear una Temp View y hacer una consulta SQL agrupada
ventas_df.createOrReplaceTempView("ventas_view")

spark.sql("""
    SELECT zona, SUM(importe) AS total_importe
    FROM ventas_view
    GROUP BY zona
""").show()

# Parte B: Subconsulta sin crear múltiples vistas
spark.sql("""
    SELECT zona, 
           COUNT(venta_id) AS num_ventas_mayores_100, 
           AVG(importe) AS media_importe
    FROM (
        SELECT * FROM ventas_view WHERE importe > 100
    ) sub
    GROUP BY zona
""").show()



+-----+-------------+
| zona|total_importe|
+-----+-------------+
|Norte|        360.0|
|  Sur|        330.0|
+-----+-------------+

+-----+----------------------+-------------+
| zona|num_ventas_mayores_100|media_importe|
+-----+----------------------+-------------+
|Norte|                     2|        135.0|
|  Sur|                     1|        200.0|
+-----+----------------------+-------------+

